# Behavior decoding judge for BSS-cleaned calcium traces

This notebook tests whether BSS-cleaned extracted traces preserve behavior-decodable structure from the raw extracted traces. The key comparisons are within-version decoding and transfer decoding between raw and cleaned trace representations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(p for p in candidates if (p / "pyproject.toml").exists() and (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from behavior_decoding import (
    load_trace_variants,
    make_behavior_targets,
    plot_behavior_preservation_summary,
    plot_behavior_trace_evidence,
    run_decoding_experiment,
    save_result,
)

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})
PROJECT_ROOT

## Configuration

In [ ]:
# Supervised behavior decoding is ONLY for v2a-RSNs datasets.
# Motorneurons remains an unsupervised-only analysis and must not be used here.
DATASET_GROUP = "v2a-RSNs"
DATA_NAME = "220127_F4_run2_fluorescence"
RAW_TRACE_CANDIDATES = [
    PROJECT_ROOT / "data" / DATASET_GROUP / "new_run2_844ROI" / "fluo_signals_no_NaN.npy",
    PROJECT_ROOT
    / "data"
    / DATASET_GROUP
    / "new_data_09112022"
    / "220127_F4_run2"
    / "220127_F4_F4_run2_after_dec_cells_fluorescence_signals.npy",
    PROJECT_ROOT
    / "data"
    / DATASET_GROUP
    / "new_data_09112022"
    / "220210_F1_run6"
    / "220127_F4_F4_run2_after_dec_cells_fluorescence_signals.npy",
]
RAW_TRACE_PATH = next((path for path in RAW_TRACE_CANDIDATES if path.exists()), None)
if RAW_TRACE_PATH is None:
    raise FileNotFoundError(
        "No supported v2a raw trace file found. Update RAW_TRACE_CANDIDATES in this cell."
    )
DATASET_DIR = RAW_TRACE_PATH.parent
RAW_NAME = RAW_TRACE_PATH.name

# Use the matching v2a tail-angle target only (no cross-dataset fallback).
TAIL_ANGLE_PATH = (
    PROJECT_ROOT
    / "data"
    / "v2a-RSNs"
    / "new_data_09112022"
    / "220127_F4_run2"
    / "220127_F4_F4_run2_after_dec_tail_angle.npy"
)
if not TAIL_ANGLE_PATH.exists():
    raise FileNotFoundError(
        f"Expected v2a tail-angle file not found: {TAIL_ANGLE_PATH}. "
        "Set TAIL_ANGLE_PATH to the matching v2a behavior recording before running."
    )

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "behavior_decoding" / DATASET_GROUP / DATA_NAME
CLEANED_ROOT_CANDIDATES = [
    # Current canonical layout from clustering_methods.ipynb:
    # outputs/linear/<dataset_group>/<data_name>/<method>/cleaned/cleaned_*.npy
    PROJECT_ROOT / "outputs" / "linear" / DATASET_GROUP / DATA_NAME,
    # Legacy clustered layout kept for older saved runs.
    PROJECT_ROOT / "outputs" / "linear" / DATASET_GROUP / "clustered" / DATASET_GROUP / DATA_NAME,
    PROJECT_ROOT / "outputs" / "linear_v2a-RSNs_clustered" / DATASET_GROUP / DATA_NAME,
    PROJECT_ROOT / "outputs" / "clustered" / DATASET_GROUP / DATA_NAME,
    PROJECT_ROOT / "outputs" / "cleaned_timeseries" / "linear",
]
CLEANED_ROOT = next((path for path in CLEANED_ROOT_CANDIDATES if path.exists()), CLEANED_ROOT_CANDIDATES[0])

CLEANED_GLOB = "cleaned_*.npy"
METHOD_GLOB = "*"
# Optional strict method filter. Keep empty tuple () to include all discovered methods.
METHODS_TO_INCLUDE = ("fastica", "infomax")
# When incremental variants are present, prefer them over canonical method-level cleaned files.
PREFER_INCREMENTAL_VARIANTS = True
INCREMENTAL_SELECTION_PREFIX = "keep_top_"
LAGS = (0, 1, 2)
TARGET_SHIFT = 0
N_SPLITS = 5
GAP = 3
RIDGE_ALPHA = 10.0
BOUT_QUANTILE = 0.75
SMOOTH_WINDOW = 3

if not RAW_TRACE_PATH.exists():
    raise FileNotFoundError(
        f"Raw v2a trace file not found: {RAW_TRACE_PATH}. "
        "This notebook should only run when the matching v2a dataset is available."
    )

DATASET_DIR, TAIL_ANGLE_PATH, OUTPUT_DIR, CLEANED_ROOT


## Load raw and cleaned traces

In [ ]:
variants = load_trace_variants(
    DATASET_DIR,
    raw_name=RAW_NAME,
    cleaned_glob=CLEANED_GLOB,
    cleaned_root=CLEANED_ROOT,
    dataset_name=DATA_NAME,
    method_glob=METHOD_GLOB,
)
if METHODS_TO_INCLUDE:
    keep = set(METHODS_TO_INCLUDE)
    variants = [
        v for v in variants if v.name == "raw" or v.name.split("/", 1)[0] in keep
    ]

raw_variants = [v for v in variants if v.name == "raw"]
if not raw_variants:
    raise ValueError("Raw trace variant was not found.")
raw_variant = raw_variants[0]
cleaned_variants = [v for v in variants if v.name != "raw"]

if PREFER_INCREMENTAL_VARIANTS:
    by_method = {}
    for variant in cleaned_variants:
        method_name = variant.name.split("/", 1)[0]
        by_method.setdefault(method_name, []).append(variant)

    preferred = []
    for method_name, method_variants in sorted(by_method.items()):
        incremental = [
            variant for variant in method_variants if f"/{INCREMENTAL_SELECTION_PREFIX}" in variant.name
        ]
        preferred.extend(incremental if incremental else method_variants)
    cleaned_variants = preferred


def _variant_sort_key(variant):
    parts = variant.name.split("/")
    method_name = parts[0] if parts else variant.name
    selection_id = (
        parts[1]
        if len(parts) >= 3 and parts[1].startswith(INCREMENTAL_SELECTION_PREFIX)
        else ""
    )
    clusters_kept = np.nan
    if selection_id:
        try:
            clusters_kept = int(selection_id.replace(INCREMENTAL_SELECTION_PREFIX, ""))
        except ValueError:
            clusters_kept = np.nan
    clusters_sort = int(clusters_kept) if np.isfinite(clusters_kept) else 10**9
    return (method_name, clusters_sort, variant.name)


cleaned_variants = sorted(cleaned_variants, key=_variant_sort_key)
variants = [raw_variant, *cleaned_variants]

if len(variants) < 2:
    raise ValueError(
        "No cleaned variants selected. Check CLEANED_ROOT and METHODS_TO_INCLUDE."
    )

trace_table = pd.DataFrame(
    {
        "version": [v.name for v in variants],
        "path": [str(v.path.relative_to(PROJECT_ROOT)) for v in variants],
        "shape": [v.traces.shape for v in variants],
    }
)
trace_table["method"] = trace_table["version"].map(
    lambda name: "raw" if name == "raw" else name.split("/", 1)[0]
)
trace_table["selection_id"] = trace_table["version"].map(
    lambda name: (
        name.split("/")[1]
        if name != "raw"
        and len(name.split("/")) >= 3
        and name.split("/")[1].startswith(INCREMENTAL_SELECTION_PREFIX)
        else None
    )
)
trace_table["clusters_kept"] = pd.to_numeric(
    trace_table["selection_id"].fillna("").str.replace(INCREMENTAL_SELECTION_PREFIX, "", regex=False),
    errors="coerce",
)
trace_table["is_incremental"] = trace_table["selection_id"].notna()
trace_table


In [ ]:
raw = variants[0].traces
trace_table

## Align tail behavior to calcium frames

In [ ]:
tail_angle = np.load(TAIL_ANGLE_PATH)
targets = make_behavior_targets(
    tail_angle,
    n_frames=raw.shape[0],
    bout_quantile=BOUT_QUANTILE,
    smooth_window=SMOOTH_WINDOW,
)

pd.Series(
    {
        "tail_samples": tail_angle.shape[0],
        "calcium_frames": raw.shape[0],
        "samples_per_calcium_frame": tail_angle.shape[0] / raw.shape[0],
        "bout_threshold": targets.bout_threshold,
        "bout_fraction": targets.bout_state.mean(),
    }
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
frames = np.arange(raw.shape[0])
axes[0].plot(frames, targets.angle, color="black", lw=0.8)
axes[0].set_ylabel("tail angle")
axes[1].plot(frames, targets.vigor, color="tab:blue", lw=0.9)
axes[1].axhline(targets.bout_threshold, color="tab:red", ls="--", lw=1)
axes[1].set_ylabel("tail vigor")
axes[2].plot(frames, targets.bout_state, color="tab:green", lw=0.9)
axes[2].set_ylabel("bout state")
axes[2].set_xlabel("calcium frame")
fig.suptitle("Tail behavior aligned to calcium frames")
fig.tight_layout(rect=(0, 0, 1, 0.97))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "tail_behavior_aligned_to_calcium_frames.png", dpi=300, bbox_inches="tight")
# fig

## Paper-ready trace and behavior evidence

This panel can be generated before running the decoding experiment. It selects a representative neuron by a stated rule and aligns raw/denoised traces to the behavior window with the strongest bout structure.

In [ ]:
trace_evidence_fig, trace_evidence_axes = plot_behavior_trace_evidence(
    variants,
    targets,
    window_size=600,
)

## Run decoding experiments

In [ ]:
common_kwargs = dict(
    variants=variants,
    lags=LAGS,
    target_shift=TARGET_SHIFT,
    n_splits=N_SPLITS,
    gap=GAP,
    ridge_alpha=RIDGE_ALPHA,
    include_transfer=True,
    include_null=True,
)

vigor_result = run_decoding_experiment(
    target=targets.vigor,
    target_name="tail_vigor",
    task="regression",
    **common_kwargs,
)

bout_result = run_decoding_experiment(
    target=targets.bout_state,
    target_name="bout_state",
    task="classification",
    **common_kwargs,
)

metrics = pd.concat([vigor_result.metrics, bout_result.metrics], ignore_index=True)
summary = pd.concat([vigor_result.summary, bout_result.summary], ignore_index=True)
summary.head()

## Paper-ready decoding and preservation summary

Run this after the decoding experiment. It shows fold-level within-decoding, raw-to-denoised transfer, and the Behavioral Preservation Index used to rank methods.

In [ ]:
preservation_fig, preservation_axes, behavior_scorecard = plot_behavior_preservation_summary(
    metrics,
    summary,
    variants=variants,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
preservation_fig.savefig(
    OUTPUT_DIR / "behavior_preservation_summary.png",
    dpi=300,
    bbox_inches="tight",
)
behavior_scorecard

In [ ]:
variant_meta = trace_table.rename(columns={"version": "variant"})[
    ["variant", "method", "selection_id", "clusters_kept", "is_incremental"]
].copy()

within_summary = summary[summary["comparison"] == "within"].copy().rename(
    columns={"test_version": "variant"}
)
within_summary = within_summary.merge(variant_meta, on="variant", how="left")
within_summary["score"] = np.where(
    within_summary["task"] == "regression",
    within_summary["r2_mean"],
    within_summary["balanced_accuracy_mean"],
)
within_summary["score_metric"] = np.where(
    within_summary["task"] == "regression",
    "r2_mean",
    "balanced_accuracy_mean",
)

raw_baseline = (
    within_summary[within_summary["variant"] == "raw"][["target", "task", "score"]]
    .rename(columns={"score": "raw_score"})
)

cluster_progress_table = (
    within_summary[within_summary["is_incremental"]]
    .merge(raw_baseline, on=["target", "task"], how="left")
    .sort_values(["target", "task", "method", "clusters_kept"])
    .copy()
)
cluster_progress_table["delta_vs_prev"] = cluster_progress_table.groupby(
    ["target", "task", "method"]
)["score"].diff()
cluster_progress_table["delta_vs_raw"] = (
    cluster_progress_table["score"] - cluster_progress_table["raw_score"]
)
cluster_progress_table = cluster_progress_table[
    [
        "target",
        "task",
        "method",
        "selection_id",
        "clusters_kept",
        "score_metric",
        "score",
        "delta_vs_prev",
        "delta_vs_raw",
        "variant",
    ]
].reset_index(drop=True)

if cluster_progress_table.empty:
    print(
        "No incremental keep-top variants detected. "
        "Run clustering_methods.ipynb incremental saves or disable PREFER_INCREMENTAL_VARIANTS."
    )
else:
    progress_plot_targets = [
        ("tail_vigor", "Tail vigor (within, R2)"),
        ("bout_state", "Bout state (within, balanced accuracy)"),
    ]
    cluster_progress_fig, axes = plt.subplots(1, len(progress_plot_targets), figsize=(6 * len(progress_plot_targets), 4))
    if len(progress_plot_targets) == 1:
        axes = [axes]

    for ax, (target_name, title) in zip(axes, progress_plot_targets):
        target_rows = cluster_progress_table[cluster_progress_table["target"] == target_name]
        for method_name, group in target_rows.groupby("method"):
            group = group.sort_values("clusters_kept")
            ax.plot(
                group["clusters_kept"],
                group["score"],
                marker="o",
                lw=1.5,
                label=method_name,
            )

        raw_row = raw_baseline[
            (raw_baseline["target"] == target_name)
            & (raw_baseline["task"] == ("regression" if target_name == "tail_vigor" else "classification"))
        ]
        if not raw_row.empty and np.isfinite(raw_row["raw_score"].iloc[0]):
            ax.axhline(
                raw_row["raw_score"].iloc[0],
                color="black",
                linestyle="--",
                linewidth=1,
                label="raw baseline",
            )

        ax.set_title(title)
        ax.set_xlabel("Clusters kept")
        ax.set_ylabel("Score")
        ax.grid(alpha=0.25)
        ax.legend(loc="best")

    cluster_progress_fig.tight_layout()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cluster_progress_fig.savefig(
        OUTPUT_DIR / "cluster_progression_summary.png",
        dpi=300,
        bbox_inches="tight",
    )

cluster_progress_table


## Supplementary prediction traces for one fold

In [ ]:
predictions = pd.concat([vigor_result.predictions, bout_result.predictions], ignore_index=True)
example_version = variants[1].name if len(variants) > 1 else "raw"
fold = 0
plot_df = predictions.query(
    "target == 'tail_vigor' and fold == @fold and "
    "((comparison == 'within' and test_version == 'raw') or "
    "(comparison == 'transfer_raw_to_clean' and test_version == @example_version))"
).copy()

fig, ax = plt.subplots(figsize=(13, 4))
for label, group in plot_df.groupby(["comparison", "test_version"]):
    ax.plot(group["time_index"], group["y_pred"], lw=1, label=f"pred {label[0]}:{label[1]}")
truth = plot_df.drop_duplicates("time_index").sort_values("time_index")
ax.plot(truth["time_index"], truth["y_true"], color="black", lw=1.4, label="true tail vigor")
ax.set_title(f"Tail-vigor predictions, fold {fold}")
ax.set_xlabel("calcium frame")
ax.set_ylabel("tail vigor")
ax.legend();

## Save outputs

In [ ]:
save_result(vigor_result, OUTPUT_DIR, "tail_vigor")
save_result(bout_result, OUTPUT_DIR, "bout_state")
metrics.to_csv(OUTPUT_DIR / "all_fold_metrics.csv", index=False)
summary.to_csv(OUTPUT_DIR / "all_summary.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "all_predictions.csv", index=False)

if "cluster_progress_table" in globals() and isinstance(cluster_progress_table, pd.DataFrame):
    cluster_progress_table.to_csv(OUTPUT_DIR / "cluster_progression_summary.csv", index=False)
if "behavior_scorecard" in globals():
    behavior_scorecard.to_csv(OUTPUT_DIR / "behavior_preservation_scorecard.csv", index=False)
if "trace_evidence_fig" in globals():
    trace_evidence_fig.savefig(OUTPUT_DIR / "paper_trace_behavior_evidence.png", dpi=300, bbox_inches="tight")
if "preservation_fig" in globals():
    preservation_fig.savefig(OUTPUT_DIR / "paper_behavior_preservation_summary.png", dpi=300, bbox_inches="tight")
if "cluster_progress_fig" in globals():
    cluster_progress_fig.savefig(OUTPUT_DIR / "paper_cluster_progression_summary.png", dpi=300, bbox_inches="tight")

OUTPUT_DIR
